**Import libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

**read Artifact**


In [ ]:
train_df = pd.read_parquet('train.parquet')

In [ ]:
#config data

artifact_dir = Path('eda_artifacts')
artifact_dir.mkdir(exist_ok=True)

**Data types, shape, and memory**

In [ ]:
# data types map (numerical, categorical, date, text, ID)
column_types_map = {
    'order_id': 'ID',
    'customer_id': 'ID',
    'customer_unique_id': 'ID',
    'order_status': 'categorical',
    'payment_types': 'categorical',
    'customer_city': 'categorical',
    'customer_state': 'categorical',
    'customer_zip_code_prefix': 'categorical',
    'seller_states': 'categorical',
    'seller_zip_codes': 'categorical',
    'order_purchase_timestamp': 'date',
    'order_approved_at': 'date',
    'order_delivered_carrier_date': 'date',
    'order_delivered_customer_date': 'date',
    'order_estimated_delivery_date': 'date',
    'total_price': 'numerical',
    'total_freight': 'numerical',
    'num_items': 'numerical',
    'num_sellers': 'numerical',
    'num_products': 'numerical',
    'total_weight': 'numerical',
    'num_photos': 'numerical',
    'avg_distance_km': 'numerical',
    'max_distance_km': 'numerical',
    'min_distance_km': 'numerical',
    'total_payment_value': 'numerical',
    'num_payment_sequential': 'numerical',
    'is_late': 'numerical'
}

mapping_df = pd.DataFrame(
    list(column_types_map.items()),
    columns=['Column', 'Semantic_Type']
 )

print(mapping_df)

In [ ]:
#shape of the training data
print(f"Training Data Shape: {train_df.shape}")

In [ ]:
#memory usage
memory_usage = train_df.memory_usage(deep=True).sum() / (1024 **2)  # Convert bytes to MB
print(f"Memory Usage: {memory_usage:.2f} MB")

**Missing values**

In [ ]:
# Missing values analysis
missing_df = train_df.isnull().sum().reset_index()
missing_df.columns = ['Column', 'Missing_Count']
missing_df['Missing_Percentage'] = missing_df['Missing_Count'] / len(train_df) * 100

# Only meaningful missing values in the current dataset
meaningful_missing_columns = {
    'order_approved_at': 'Order not approved.',
    'order_delivered_carrier_date': 'Order not shipped.',
    'total_price': 'Order has no item records.',
    'total_freight': 'Order has no item records.',
    'num_items': 'Order has no item records.',
    'num_sellers': 'Order has no seller records.',
    'num_products': 'Order has no product records.',
    'total_weight': 'Product weight not recorded.',
    'num_photos': 'Product photo count not recorded.',
    'avg_distance_km': 'Seller/customer distance unavailable.',
    'max_distance_km': 'Seller/customer distance unavailable.',
    'min_distance_km': 'Seller/customer distance unavailable.'
}

missing_df['Meaningful_Missing'] = (
    missing_df['Column'].map(meaningful_missing_columns).fillna('Not meaningful')
 )
print(missing_df)

**Numerical analysis**

In [ ]:
numeric_columns = [
    column for column, semantic_type in column_types_map.items()
    if semantic_type == 'numerical' and column in train_df.columns and column != 'is_late'
    ]

numeric_summary = train_df[numeric_columns].describe().T
numeric_summary['median'] = train_df[numeric_columns].median()
numeric_summary['skewness'] = train_df[numeric_columns].skew()
numeric_summary['range'] = numeric_summary['max'] - numeric_summary['min']
numeric_summary['outlier_count'] = 0

for column in numeric_columns:
    q1 = train_df[column].quantile(0.25)
    q3 = train_df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    numeric_summary.loc[column, 'outlier_count'] = int(
        ((train_df[column] < lower_bound) | (train_df[column] > upper_bound)).sum()
    )

numeric_summary['outlier_percentage'] = numeric_summary['outlier_count'] / len(train_df) * 100
print('--- Numerical Summary ---')
print(numeric_summary.round(3).to_string())

non_negative_columns = [
    column for column in numeric_columns
    if any(keyword in column.lower() for keyword in ['price', 'freight', 'items', 'payment', 'weight', 'photos', 'seller', 'product', 'distance'])
    ]
invalid_ranges = {
    column: int((train_df[column] < 0).sum())
    for column in non_negative_columns
    }
print('\n--- Values Outside Meaningful Ranges ---')
for column, invalid_count in invalid_ranges.items():
    print(f'{column}: {invalid_count} negative values')

fig, axes = plt.subplots(2, len(numeric_columns), figsize=(4 * len(numeric_columns), 7), squeeze=False)
for index, column in enumerate(numeric_columns):
    sns.histplot(train_df[column].dropna(), kde=True, ax=axes[0, index])
    axes[0, index].set_title(f'{column} distribution')
    sns.boxplot(x=train_df[column], ax=axes[1, index])
    axes[1, index].set_title(f'{column} outliers')
plt.tight_layout()
plt.savefig(artifact_dir / 'numerical_distributions_and_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

**Categorical analysis**

In [ ]:
categorical_columns = [
    column for column, semantic_type in column_types_map.items()
    if semantic_type == 'categorical' and column in train_df.columns
    ]
categorical_summary = []
messy_values = []

for column in categorical_columns:
    values = train_df[column].dropna().astype(str)
    counts = values.value_counts()
    normalized = values.str.strip().str.lower()
    normalized_counts = normalized.value_counts()
    categorical_summary.append({
        'column': column,
        'cardinality': int(values.nunique()),
        'missing_count': int(train_df[column].isna().sum()),
        'rare_categories_lt_1pct': int((counts / len(train_df) < 0.01).sum()),
        'normalized_duplicate_groups': int((normalized_counts > 1).sum())
    })
    for normalized_value in normalized_counts[normalized_counts > 1].index:
        original_values = sorted(values[normalized == normalized_value].unique())
        if len(original_values) > 1:
            messy_values.append({
                'column': column,
                'normalized_value': normalized_value,
                'original_values': original_values
            })

categorical_summary_df = pd.DataFrame(categorical_summary)
messy_values_df = pd.DataFrame(messy_values)
print(categorical_summary_df.to_string(index=False))
print('\nMessy or duplicated values:')
print(messy_values_df.to_string(index=False) if not messy_values_df.empty else 'None found')

for column in categorical_columns:
    counts = train_df[column].fillna('<MISSING>').astype(str).value_counts().head(15)
    plt.figure(figsize=(8, 4))
    sns.barplot(x=counts.values, y=counts.index, hue=counts.index, legend=False, palette='viridis')
    plt.title(f'Top Categories: {column}')
    plt.tight_layout()
    plt.savefig(artifact_dir / f'categorical_{column}.png', dpi=150, bbox_inches='tight')
    plt.show()

**Relations with the label**

In [ ]:
status_relation = train_df.groupby('order_status', dropna=False)['is_late'].agg(['count', 'mean']).reset_index()
status_relation['late_rate_percent'] = status_relation['mean'] * 100
payment_relation = train_df.groupby('payment_types', dropna=False)['is_late'].agg(['count', 'mean']).reset_index()
payment_relation['late_rate_percent'] = payment_relation['mean'] * 100
cross_tab = pd.crosstab(train_df['order_status'], train_df['is_late'], normalize='index') * 100
correlation_matrix = train_df[numeric_columns + ['is_late']].corr()

print(status_relation.sort_values('count', ascending=False).to_string(index=False))
print('\n', payment_relation.sort_values('count', ascending=False).to_string(index=False))
print('\nCross-tab (%):\n', cross_tab.round(2))
print('\nCorrelations with is_late:\n', correlation_matrix['is_late'].sort_values(ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=status_relation, x='order_status', y='late_rate_percent', hue='order_status', legend=False, ax=axes[0], palette='Reds')
sns.barplot(data=payment_relation, x='payment_types', y='late_rate_percent', hue='payment_types', legend=False, ax=axes[1], palette='Oranges')
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[2])
axes[0].set_title('Late Rate by Order Status')
axes[1].set_title('Late Rate by Payment Type')
axes[2].set_title('Numerical Correlations')
for axis in axes[:2]:
    axis.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(artifact_dir / 'relations_and_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

**Date analysis**

In [ ]:
date_columns = [
    column for column, semantic_type in column_types_map.items()
    if semantic_type == 'date' and column in train_df.columns
    ]
date_df = train_df.copy()
for column in date_columns:
    date_df[column] = pd.to_datetime(date_df[column], errors='coerce')

date_df['purchase_month'] = date_df['order_purchase_timestamp'].dt.month
date_df['purchase_weekday'] = date_df['order_purchase_timestamp'].dt.day_name()
date_df['delivery_days'] = (date_df['order_delivered_customer_date'] - date_df['order_purchase_timestamp']).dt.total_seconds() / 86400
date_df['delivery_delay_days'] = (date_df['order_delivered_customer_date'] - date_df['order_estimated_delivery_date']).dt.total_seconds() / 86400

# Approximate Brazilian public holidays covered by the training period.
holiday_dates = pd.to_datetime([
    '2016-11-02', '2016-11-15', '2016-12-25',
    '2017-01-01', '2017-02-27', '2017-02-28', '2017-04-14', '2017-04-21',
    '2017-05-01', '2017-06-15', '2017-09-07', '2017-10-12', '2017-11-02',
    '2017-11-15', '2017-12-25', '2018-01-01', '2018-02-12', '2018-02-13',
    '2018-03-30', '2018-04-21', '2018-05-01', '2018-09-07'
    ])
date_df['is_holiday'] = date_df['order_purchase_timestamp'].dt.normalize().isin(holiday_dates)

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_late_rate = date_df.groupby('purchase_weekday')['is_late'].mean().reindex(weekday_order) * 100
monthly_late_rate = date_df.groupby('purchase_month')['is_late'].mean() * 100
holiday_late_rate = date_df.groupby('is_holiday')['is_late'].mean() * 100

print(f"Purchase date range: {date_df['order_purchase_timestamp'].min()} to {date_df['order_purchase_timestamp'].max()}")
print(f"Median delivery time: {date_df['delivery_days'].median():.2f} days")
print(f"Median delivery delay vs estimate: {date_df['delivery_delay_days'].median():.2f} days")
print('\nLate rate by weekday (%):')
print(weekday_late_rate.round(2).to_string())
print('\nLate rate by holiday flag (%):')
print(holiday_late_rate.round(2).to_string())

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
sns.lineplot(x=monthly_late_rate.index, y=monthly_late_rate.values, marker='o', ax=axes[0])
axes[0].set_title('Monthly Late Rate')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Late Rate (%)')
sns.barplot(x=weekday_late_rate.index, y=weekday_late_rate.values, ax=axes[1], color='#2ecc71')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_title('Late Rate by Weekday')
sns.barplot(x=holiday_late_rate.index.astype(str), y=holiday_late_rate.values, ax=axes[2], color='#9b59b6')
axes[2].set_title('Holiday Effect')
axes[2].set_xlabel('Is Holiday')
axes[2].set_ylabel('Late Rate (%)')
sns.histplot(date_df['delivery_days'].dropna(), kde=True, ax=axes[3], color='#3498db')
axes[3].set_title('Delivery Time Distribution')
axes[3].set_xlabel('Days')
plt.tight_layout()
plt.savefig(artifact_dir / 'date_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

**Geography analysis**

In [ ]:
# Distance features are already computed in train.parquet.
geo_df = train_df.copy()

geo_df['same_state'] = geo_df.apply(
    lambda row: row['customer_state'] in str(row['seller_states']).split(', '), axis=1
 )
state_late_rate = (
    geo_df.groupby('customer_state')['is_late']
    .agg(['count', 'mean'])
    .rename(columns={'mean': 'late_rate'})
    .sort_values('late_rate', ascending=False)
 )
state_late_rate['late_rate_percent'] = state_late_rate['late_rate'] * 100
same_state_late_rate = geo_df.groupby('same_state')['is_late'].mean() * 100

print('Late rate: same seller/customer state vs different state (%):')
print(same_state_late_rate.round(2))
print('\nTop customer states by late rate (min 20 orders):')
print(state_late_rate[state_late_rate['count'] >= 20].head(10).round(3))

print('\nMissing avg_distance_km:', geo_df['avg_distance_km'].isna().sum(), '/', len(geo_df))
print(geo_df[['avg_distance_km', 'max_distance_km', 'min_distance_km']].describe())

geo_df['distance_bucket'] = pd.cut(
    geo_df['avg_distance_km'], bins=[-1, 0, 100, 500, 1000, 2000, 10000],
    labels=['0', '0-100', '100-500', '500-1000', '1000-2000', '2000+']
 )
distance_late_rate = geo_df.groupby('distance_bucket', observed=True)['is_late'].agg(['count', 'mean'])
distance_late_rate['late_rate_percent'] = distance_late_rate['mean'] * 100
print('\nLate rate by avg_distance_km bucket:')
print(distance_late_rate.round(3))

distance_corr = geo_df[['avg_distance_km', 'max_distance_km', 'min_distance_km', 'is_late']].corr()['is_late']
print('\nCorrelation with is_late:')
print(distance_corr.round(3))

print('\nZIP-code cardinality:')
print('customer_zip_code_prefix:', geo_df['customer_zip_code_prefix'].nunique())
print('seller_zip_codes:', geo_df['seller_zip_codes'].nunique())

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
sns.histplot(geo_df['avg_distance_km'].dropna(), kde=True, ax=axes[0], color='#e67e22')
axes[0].set_title('Average Customer-Seller Distance (km)')
sns.barplot(x=distance_late_rate.index.astype(str), y=distance_late_rate['late_rate_percent'], ax=axes[1], color='#c0392b')
axes[1].set_title('Late Rate by Distance Bucket')
axes[1].tick_params(axis='x', rotation=45)
top_states = state_late_rate[state_late_rate['count'] >= 20].head(10)
sns.barplot(x=top_states.index, y=top_states['late_rate_percent'], ax=axes[2], color='#16a085')
axes[2].set_title('Late Rate by Customer State')
axes[2].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(artifact_dir / 'geography_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

**features selection**

In [ ]:
# Feature selection: candidate features to carry into Notebook 5 (feature engineering),
# based on what the EDA above actually showed.
# For each feature: what it represents, why it was picked, and how to derive it if it
# is not a raw column (i.e. it needs to be computed from other columns first).

feature_selection = [
    {
        'feature': 'total_price',
        'represents': 'Sum of item prices for the order.',
        'why_selected': 'Right-skewed with real outliers (see Numerical analysis); higher-value orders may ship differently.',
        'derived': False,
        'how_to_derive': 'Raw column from Notebook 1 (sum of order_items.price per order_id).'
    },
    {
        'feature': 'total_freight',
        'represents': 'Sum of freight/shipping cost for the order.',
        'why_selected': 'Freight cost is a proxy for shipping method/distance and showed a non-trivial correlation with is_late.',
        'derived': False,
        'how_to_derive': 'Raw column from Notebook 1 (sum of order_items.freight_value per order_id).'
    },
    {
        'feature': 'num_items',
        'represents': 'Number of items in the order.',
        'why_selected': 'More items can mean more picking/packing time and more chances for delay.',
        'derived': False,
        'how_to_derive': 'Raw column from Notebook 1 (count of order_items rows per order_id).'
    },
    {
        'feature': 'num_sellers',
        'represents': 'Number of distinct sellers fulfilling the order.',
        'why_selected': 'Multi-seller orders depend on every seller shipping on time, so this is a plausible risk factor for delay.',
        'derived': False,
        'how_to_derive': 'Raw column from Notebook 1 (nunique of seller_id per order_id).'
    },
    {
        'feature': 'num_products',
        'represents': 'Number of distinct products in the order.',
        'why_selected': 'Similar reasoning to num_items -- more distinct products can mean more sourcing/packing complexity.',
        'derived': False,
        'how_to_derive': 'Raw column from Notebook 1 (nunique of product_id per order_id).'
    },
    {
        'feature': 'total_weight',
        'represents': 'Total weight (grams) of all items in the order.',
        'why_selected': 'Heavier shipments can require different carriers/handling; distribution and outliers were checked in Numerical analysis.',
        'derived': False,
        'how_to_derive': 'Raw column from Notebook 1 (sum of product_weight_g per order_id).'
    },
    {
        'feature': 'avg_distance_km',
        'represents': 'Average real-world distance between the customer and the order\'s seller(s).',
        'why_selected': 'Geography analysis showed a higher late rate at greater distance buckets and a positive correlation with is_late.',
        'derived': True,
        'how_to_derive': 'Notebook 1: haversine distance per item (seller zip vs customer zip coordinates from geolocation), averaged per order_id.'
    },
    {
        'feature': 'max_distance_km',
        'represents': 'Distance to the farthest seller in the order.',
        'why_selected': 'An order is only complete once its farthest seller\'s items arrive, so this can matter more than the average.',
        'derived': True,
        'how_to_derive': 'Notebook 1: same per-item haversine distance as avg_distance_km, aggregated with max instead of mean, per order_id.'
    },
    {
        'feature': 'same_state',
        'represents': 'Whether at least one seller is in the same state as the customer.',
        'why_selected': 'Geography analysis showed a different late rate for same-state vs different-state shipments; a cheap, low-cardinality complement to the distance features.',
        'derived': True,
        'how_to_derive': 'Notebook 5: customer_state in seller_states.split(\', \') -- boolean, computed from two existing columns, no new data needed.'
    },
    {
        'feature': 'delivery_delay_days (only for reference, not usable as a model input)',
        'represents': 'Difference in days between the actual and estimated delivery date -- this is how is_late itself is built.',
        'why_selected': 'Not selected as a feature: it leaks the label. Listed here only to flag it must be excluded from the feature set in Notebook 5.',
        'derived': True,
        'how_to_derive': 'N/A -- excluded on purpose.'
    },
    {
        'feature': 'purchase_month',
        'represents': 'Calendar month the order was purchased in.',
        'why_selected': 'Date analysis showed the late rate moves across months (seasonality, e.g. holiday shopping peaks).',
        'derived': True,
        'how_to_derive': 'Notebook 5: order_purchase_timestamp.dt.month.'
    },
    {
        'feature': 'purchase_weekday',
        'represents': 'Day of week the order was purchased.',
        'why_selected': 'Date analysis showed a weekday effect on the late rate.',
        'derived': True,
        'how_to_derive': 'Notebook 5: order_purchase_timestamp.dt.day_name() (or dt.dayofweek for a numeric/cyclical encoding).'
    },
    {
        'feature': 'is_holiday',
        'represents': 'Whether the order was purchased on a Brazilian public holiday.',
        'why_selected': 'Date analysis showed a different late rate on holidays vs regular days.',
        'derived': True,
        'how_to_derive': 'Notebook 5: order_purchase_timestamp.dt.normalize().isin(holiday_dates), same holiday list used in this EDA.'
    },
    {
        'feature': 'order_status',
        'represents': 'Order status at the time the data was extracted (e.g. delivered, canceled).',
        'why_selected': 'Relations analysis showed a different late rate by status; almost all rows are \'delivered\' after the Notebook 2 filtering, but the few exceptions are informative.',
        'derived': False,
        'how_to_derive': 'Raw column from orders, already in train_df.'
    },
    {
        'feature': 'payment_types',
        'represents': 'Payment method(s) used for the order (e.g. credit_card, boleto, voucher).',
        'why_selected': 'Relations analysis showed the late rate varies by payment type.',
        'derived': False,
        'how_to_derive': 'Raw column from Notebook 1 (aggregated unique payment_type values per order_id); needs multi-hot encoding in Notebook 5 since an order can have more than one payment type.'
    },
    {
        'feature': 'num_payment_sequential',
        'represents': 'Number of payment installments/methods used for the order.',
        'why_selected': 'A simple numeric complement to payment_types; more installments could reflect order value or customer behavior linked to delay.',
        'derived': False,
        'how_to_derive': 'Raw column from Notebook 1 (count of order_payments rows per order_id).'
    },
    {
        'feature': 'customer_state',
        'represents': 'State of the customer.',
        'why_selected': 'Geography analysis showed the late rate differs meaningfully across states.',
        'derived': False,
        'how_to_derive': 'Raw column from customers, already in train_df. High-cardinality-safe (27 Brazilian states), unlike zip prefix.'
    }
]

feature_selection_df = pd.DataFrame(feature_selection)
print(feature_selection_df[['feature', 'derived']].to_string(index=False))

feature_selection_df.to_csv(artifact_dir / 'feature_selection.csv', index=False)
print(f"\nSaved feature selection to: {(artifact_dir / 'feature_selection.csv').resolve()}")


I chose **Random Forest** because it handles mixed numerical/categorical features without heavy preprocessing, is robust to the skewed distributions and outliers found in EDA, and supports class weighting for the ~12:1 imbalance. It also provides feature importances, letting us confirm whether the signals found in EDA (distance, freight, num_sellers) actually drive predictions.

**EDA artifacts and findings**

In [ ]:
categorical_summary_df.to_csv(artifact_dir / 'categorical_summary.csv', index=False)
messy_values_df.to_csv(artifact_dir / 'messy_values.csv', index=False)
numeric_summary.to_csv(artifact_dir / 'numeric_summary.csv')
missing_df.to_csv(artifact_dir / 'missing_summary.csv', index=False)
status_relation.to_csv(artifact_dir / 'status_relation.csv', index=False)
payment_relation.to_csv(artifact_dir / 'payment_relation.csv', index=False)
cross_tab.to_csv(artifact_dir / 'status_label_crosstab.csv')
correlation_matrix.to_csv(artifact_dir / 'correlation_matrix.csv')
distance_late_rate.to_csv(artifact_dir / 'distance_late_rate.csv')
distance_corr.to_csv(artifact_dir / 'distance_corr.csv')

findings = [
    'EDA uses train.parquet only; test data was not opened.',
    f'Training shape: {train_df.shape[0]:,} rows and {train_df.shape[1]} columns.',
    'Customer city and ZIP prefixes have high cardinality and need careful encoding.',
    f'Median delivery time: {date_df["delivery_days"].median():.2f} days.',
    'Numerical features are right-skewed and contain IQR outliers.',
    f'Holiday late rate: {holiday_late_rate.get(True, float("nan")):.2f}% versus {holiday_late_rate.get(False, float("nan")):.2f}% on non-holidays.',
    'Longer customer-seller distance is associated with a higher late rate.',
    'Use these findings to guide imputation, encoding, outlier treatment, and model selection.'
    ]
(artifact_dir / 'findings_summary.txt').write_text('\n'.join(findings), encoding='utf-8')
print(f'Saved EDA artifacts to: {artifact_dir.resolve()}')